# SAP News Intelligence — Embedding & Retrieval Pipeline

This notebook builds a retrieval-augmented pipeline over SAP-related news articles:

1. Load the scraped/cleaned articles
2. **Data cleaning** (dedupe, normalize text, drop broken rows, parse dates)
3. Chunk article text for embedding
4. Generate sentence embeddings (cached to disk so re-runs are fast)
5. Build a FAISS similarity index
6. Retrieve relevant chunks for a query
7. Run an LLM "opportunity agent" over the retrieved context

**Portability note:** paths are resolved relative to the project root (or a
`PROJECT_ROOT` environment variable) instead of being hardcoded to a specific
machine, so this notebook runs unmodified on any computer / grading environment.


## 1. Setup & configuration

In [ ]:
import os
import re
import unicodedata
from pathlib import Path

import numpy as np
import pandas as pd
from dotenv import load_dotenv

load_dotenv()  # reads a local .env file if present (API keys, overrides, etc.)

# Resolve the project root instead of hardcoding a machine-specific path.
# Override by setting PROJECT_ROOT in your environment / .env file.
PROJECT_ROOT = Path(os.getenv("PROJECT_ROOT", Path.cwd())).resolve()
DATA_DIR = PROJECT_ROOT / "data"
DATA_DIR.mkdir(parents=True, exist_ok=True)

RAW_DATA_PATH = DATA_DIR / "clean_data.json"
CHUNKED_DATA_PATH = DATA_DIR / "chunked_data.json"
EMBEDDINGS_PATH = DATA_DIR / "chunk_embeddings.npy"
FAISS_INDEX_PATH = DATA_DIR / "sap_intelligence.index"

CHUNK_SIZE = 1000
CHUNK_OVERLAP = 200
EMBEDDING_MODEL_NAME = "BAAI/bge-small-en-v1.5"
TOP_K_DEFAULT = 10
MIN_TEXT_LENGTH = 50  # filters out near-empty / broken scrapes

REQUIRED_COLUMNS = ["title", "source", "url", "published", "clean_text"]

print(f"Project root: {PROJECT_ROOT}")
print(f"Data dir:     {DATA_DIR}")

## 2. Load raw data

Wrapped in a function with a clear error message instead of a bare
`pd.read_json` call, so a missing file fails fast and explains how to fix it.

In [ ]:
def load_raw_data(path: Path) -> pd.DataFrame:
    if not path.exists():
        raise FileNotFoundError(
            f"Could not find input data at {path}. "
            "Set the PROJECT_ROOT environment variable, or place clean_data.json "
            "under ./data relative to this notebook."
        )
    df = pd.read_json(path)
    print(f"Loaded {len(df)} rows, {df.shape[1]} columns from {path.name}")
    return df


raw_df = load_raw_data(RAW_DATA_PATH)
raw_df.head()

## 3. Data cleaning

The original notebook assumed the input was already "clean". In practice,
scraped news data usually has duplicate articles, missing fields, stray
HTML/whitespace, and inconsistent date formats — all of which hurt chunk
quality and retrieval relevance. This step:

- Drops rows missing essential fields (`clean_text`, `title`, `url`)
- Normalizes unicode characters and collapses whitespace/HTML remnants
- Drops articles that are suspiciously short (likely failed scrapes)
- De-duplicates by URL, and by `(title, source)` as a fallback
- Parses `published` into a real datetime (UTC) and flags rows that fail to parse
- Sorts chronologically and resets the index for clean, reproducible `chunk_id`s


In [ ]:
def normalize_whitespace(text: str) -> str:
    text = unicodedata.normalize("NFKC", text)
    text = re.sub(r"<[^>]+>", " ", text)   # strip stray HTML tags
    text = re.sub(r"\s+", " ", text)       # collapse whitespace / newlines
    return text.strip()


def clean_dataframe(df: pd.DataFrame) -> pd.DataFrame:
    df = df.copy()

    missing_cols = [c for c in REQUIRED_COLUMNS if c not in df.columns]
    if missing_cols:
        raise KeyError(f"Expected columns missing from input data: {missing_cols}")

    before = len(df)

    # Drop rows missing essential fields
    df = df.dropna(subset=["clean_text", "title", "url"])

    # Normalize text fields
    df["title"] = df["title"].astype(str).map(normalize_whitespace)
    df["clean_text"] = df["clean_text"].astype(str).map(normalize_whitespace)
    df["source"] = df["source"].astype(str).str.strip()

    # Drop empty / too-short articles (likely failed scrapes)
    df = df[df["clean_text"].str.len() >= MIN_TEXT_LENGTH]

    # De-duplicate: same URL, or same (title, source) pair
    df = df.drop_duplicates(subset=["url"])
    df = df.drop_duplicates(subset=["title", "source"])

    # Standardize the published date; keep rows even if parsing fails, but flag them
    df["published"] = pd.to_datetime(df["published"], errors="coerce", utc=True)
    n_unparsed = int(df["published"].isna().sum())
    if n_unparsed:
        print(f"Warning: {n_unparsed} rows have an unparseable 'published' date.")

    df = df.sort_values("published", na_position="last").reset_index(drop=True)

    after = len(df)
    print(f"Cleaning removed {before - after} rows ({before} -> {after}).")
    return df


clean_df = clean_dataframe(raw_df)
clean_df.shape

## 4. Chunking

Same `RecursiveCharacterTextSplitter` settings as the original, now wrapped in functions and run on the *cleaned* dataframe.

In [ ]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

def build_splitter(chunk_size=CHUNK_SIZE, chunk_overlap=CHUNK_OVERLAP):
    return RecursiveCharacterTextSplitter(chunk_size=chunk_size, chunk_overlap=chunk_overlap)


def chunk_dataframe(df: pd.DataFrame, splitter) -> pd.DataFrame:
    records = []
    for row_id, row in df.iterrows():
        text_chunks = splitter.split_text(row["clean_text"])
        for i, chunk in enumerate(text_chunks):
            records.append({
                "chunk_id": f"{row_id}_{i}",
                "text": chunk,
                "title": row["title"],
                "source": row["source"],
                "url": row["url"],
                "published": row["published"],
            })
    chunks_df = pd.DataFrame(records)
    chunks_df["chunk_length"] = chunks_df["text"].str.len()
    return chunks_df


splitter = build_splitter()
chunks_df = chunk_dataframe(clean_df, splitter)
print(chunks_df.shape)
chunks_df["chunk_length"].describe()

In [ ]:
chunks_df.to_json(CHUNKED_DATA_PATH, orient="records", indent=4, date_format="iso")
print(f"Saved chunked data to {CHUNKED_DATA_PATH}")

## 5. Embeddings

Same embedding model as the original (`BAAI/bge-small-en-v1.5`), but:
- Wrapped in functions
- **Cached to disk** — re-running the notebook won't recompute embeddings for
  unchanged data, which matters once you have hundreds/thousands of chunks


In [ ]:
from sentence_transformers import SentenceTransformer

def load_embedding_model(model_name=EMBEDDING_MODEL_NAME):
    return SentenceTransformer(model_name)


def compute_or_load_embeddings(texts, model, cache_path: Path = EMBEDDINGS_PATH) -> np.ndarray:
    if cache_path.exists():
        cached = np.load(cache_path)
        if cached.shape[0] == len(texts):
            print(f"Loaded cached embeddings from {cache_path} {cached.shape}.")
            return cached
        print("Cached embeddings size doesn't match current data; recomputing.")

    embeddings = model.encode(
        texts, show_progress_bar=True, normalize_embeddings=True
    ).astype("float32")
    np.save(cache_path, embeddings)
    print(f"Computed and cached embeddings {embeddings.shape} at {cache_path}.")
    return embeddings


embedding_model = load_embedding_model()
texts = chunks_df["text"].tolist()
embeddings = compute_or_load_embeddings(texts, embedding_model)
embeddings.shape

## 6. FAISS vector index

The original notebook built this index twice (redundant cells); consolidated into one function here.

In [ ]:
import faiss

def build_faiss_index(embeddings: np.ndarray) -> faiss.Index:
    dimension = embeddings.shape[1]
    index = faiss.IndexFlatIP(dimension)
    index.add(embeddings)
    return index


faiss_index = build_faiss_index(embeddings)
faiss.write_index(faiss_index, str(FAISS_INDEX_PATH))
print(f"FAISS index built with {faiss_index.ntotal} vectors and saved to {FAISS_INDEX_PATH}")

## 7. Retrieval

Same behavior as the original `retrieve()`, plus similarity scores in the output and configurable de-duplication.

In [ ]:
def retrieve(query: str, k: int = TOP_K_DEFAULT, dedupe_by_title: bool = True) -> pd.DataFrame:
    query_embedding = embedding_model.encode([query], normalize_embeddings=True).astype("float32")
    scores, indices = faiss_index.search(query_embedding, k)

    results = chunks_df.iloc[indices[0]].copy()
    results["score"] = scores[0]

    if dedupe_by_title:
        results = results.drop_duplicates(subset=["title"])

    return results[["title", "source", "published", "score", "text"]].reset_index(drop=True)


retrieve("What AI initiatives is SAP pursuing?")

## 8. Opportunity agent (LLM step)

The original `opportunity_agent` referenced an undefined `llm` variable. This
version adds a `get_llm()` helper that picks a provider based on whichever API
key is set in your environment (`ANTHROPIC_API_KEY` or `OPENAI_API_KEY`), or
an explicit `LLM_PROVIDER` override. Add the key(s) to a local `.env` file —
see `.env.example`.


In [ ]:
def get_llm():
    """
    Picks a chat model based on whichever API key is available.
    Set ANTHROPIC_API_KEY or OPENAI_API_KEY in a .env file, or force a
    provider explicitly with LLM_PROVIDER=anthropic|openai.
    """
    provider = os.getenv("LLM_PROVIDER", "").lower()

    if provider == "anthropic" or (not provider and os.getenv("ANTHROPIC_API_KEY")):
        from langchain_anthropic import ChatAnthropic
        return ChatAnthropic(model="claude-sonnet-4-6", temperature=0.2)

    if provider == "openai" or (not provider and os.getenv("OPENAI_API_KEY")):
        from langchain_openai import ChatOpenAI
        return ChatOpenAI(model="gpt-4o-mini", temperature=0.2)

    raise EnvironmentError(
        "No LLM credentials found. Set ANTHROPIC_API_KEY or OPENAI_API_KEY in your .env file."
    )


def opportunity_agent(context: str, llm=None) -> str:
    """Analyze retrieved context and surface business opportunities."""
    llm = llm or get_llm()

    prompt = f"""You are a strategy analyst reviewing recent SAP-related news.

Based on the context below, identify and briefly justify:
1. Revenue opportunities
2. Market opportunities
3. Partnership opportunities
4. AI opportunities

Context:
{context}
"""
    response = llm.invoke(prompt)
    return response.content

In [ ]:
# Example usage (requires an API key set in .env):
# results = retrieve("What AI initiatives is SAP pursuing?")
# context = "\n\n".join(results["text"].tolist())
# print(opportunity_agent(context))